# Tattersalls Autumn Horses in Training Sale — Data Preparation

**Notebook 01 of 3** | *TFM — Predictive Modelling of the Tattersalls Horses in Training Market*

This notebook transforms the raw Tattersalls catalogue CSV into a clean, analytically framed
dataset ready for EDA and modelling. Three invariants are enforced throughout:

1. **Reproducibility** — all transformations are deterministic and fully parameterised.
2. **Non-destructive cleaning** — original text columns are preserved; derived fields are added alongside them.
3. **Explicit outcome definitions** — `withdrawn`, `not sold`, `vendor buyback`, and
   `sold to third party` are treated as economically distinct states.

| Step | Section |
|------|---------|
| 0 | Setup & Analytical Constants |
| 1 | Data Ingestion |
| 2 | Column Standardisation |
| 3 | Data Dictionary |
| 4 | Entity Normalisation |
| 5 | Derived Features |
| 6 | Outcome Engineering |
| 7 | Temporal Sanity Check |
| 8 | Export |

In [1]:
import os
import re
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from pathlib import Path
sys.path.insert(0, str(Path('').resolve()))
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent))
import seaborn as sns
import plotly.express as px
import onspy

# Global plot settings
sns.set_style("whitegrid")
plt.rcParams['axes.formatter.limits'] = (-20, 20)
plt.rcParams['axes.formatter.useoffset'] = False
#!uv pip install -r requirements.txt --prerelease=allow

> **Design rationale:** This notebook is intentionally strict about three things before interpreting any business pattern: reproducibility, non-destructive cleaning, and explicit target definitions. The goal is not only to describe the Tattersalls market, but to identify which variables are real, which ones are proxies, and which ones would create leakage or misleading conclusions in a later modeling stage.
**Note**: To model the market accurately, we define `df_offered` by explicitly excluding 'withdrawn' lots. Withdrawn lots never faced the market and thus shouldn't be treated as failures to sell. We also strictly separate 'vendor buybacks' from actual 'sales to third parties' to avoid upward price bias.

## 0. Setup & Analytical Constants

Global imports, plot defaults, and domain constants.
The `PREMIUM_CUTOFF` (150 000 gns) defines the elite-tier threshold used throughout the TFM.
CPI values (ONS, October reference month, base 2024) enable real-price comparisons across
the full 2009–2025 window. Utility functions for bootstrapped CIs and permutation tests
are defined here once and reused in the EDA notebook.

In [2]:
pd.set_option('display.max_columns', 50)
pd.set_option('display.float_format', lambda x: f'{x:,.2f}')

# ── All imports from src/data_prep.py ──
# These were previously defined inline and duplicated here.
# Source of truth is now src/data_prep.py.

from src.data_prep import (
    PREMIUM_CUTOFF, EARLY_PERIOD, MID_PERIOD, RECENT_PERIOD,
    BASE_YEAR, cpi_october, COUNTRY_SUFFIX_RE,
    extract_country_suffix, strip_country_suffix,
    parse_numeric_series, title_from_canonical,
    normalize_root_entity, bootstrap_ci,
    bootstrap_proportion_ci, permutation_test,
    mean_annual_share_table,
)

print(f'Setup OK — PREMIUM_CUTOFF: {PREMIUM_CUTOFF:,} gns')
print(f'  CPI base year: {BASE_YEAR}')
print(f'  Periods: early={EARLY_PERIOD}, mid={MID_PERIOD}, recent={RECENT_PERIOD}')

Setup OK — PREMIUM_CUTOFF: 150,000 gns
  CPI base year: 2024
  Periods: early=(2009, 2015), mid=(2016, 2020), recent=(2021, 2025)


## 1. Data Ingestion

Load the raw Tattersalls CSV (26 076 lots, 2009–2025).
The file uses European numeric formatting (`90.000` for ninety thousand) and
mixed-language column names — both are resolved in subsequent steps.

In [3]:
autumn_horses_df = pd.read_csv('../data/Autumn Horses In Training Sale 2009-2024.csv')
autumn_horses_df.info()
autumn_horses_df.head()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 26076 entries, 0 to 26075
Data columns (total 24 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   Day             26076 non-null  int64  
 1   Lot             26076 non-null  object 
 2   Name            26076 non-null  object 
 3   Sex             26054 non-null  object 
 4   Colour          26053 non-null  object 
 5   Sire            26049 non-null  object 
 6   Dam             26049 non-null  object 
 7   Year Foaled     26054 non-null  float64
 8   Date Foaled     26054 non-null  object 
 9   Grandsire       26024 non-null  object 
 10  Damsire         26024 non-null  object 
 11  Covered by      6 non-null      object 
 12  Consignor       26049 non-null  object 
 13  Purchaser       26076 non-null  object 
 14  Price (gns)     18695 non-null  object 
 15  Stabling        454 non-null    object 
 16  Año Subasta     26076 non-null  int64  
 17  Nombre Subasta  26076 non-null 

,Day,Lot,Name,Sex,Colour,Sire,Dam,Year Foaled,Date Foaled,Grandsire,Damsire,Covered by,Consignor,Purchaser,Price (gns),Stabling,Año Subasta,Nombre Subasta,Price (€),ORIG,SIRE_N,DAM_N,SIREDAM_N,BREEDER_N
0,3,1067,Qirat (GB),G,Ch,Showcasing (GB),Emulous (GB),"2,021.00",22/2/2021,Oasis Dream (GB),Dansili (GB),NaN,Juddmonte,Lot Withdrawn,NaN,NaN,2024,Autumn Horses In Training Sale 2024,-,2021SHOWCASINGEMULOUS,SHOWCASING,EMULOUS,DANSILI,JUDDMONTE
1,2,639,Gifted Master (IRE),G,B,Kodiac (GB),Shobobb (GB),"2,013.00",3/4/2013,Danehill (USA),Shamardal (USA),NaN,The Castlebridge Consignment,Lot Withdrawn,NaN,NaN,2019,Autumn Horses in Training Sale 2019,-,2013KODIACSHOBOBB,KODIAC,SHOBOBB,SHAMARDAL,THE CASTLEBRIDGE CONSIGNMENT
2,2,568,Commanche Falls (GB),G,Br,Lethal Force (IRE),Joyeaux (GB),"2,017.00",28/4/2017,Dark Angel (IRE),Mark of Esteem (IRE),NaN,Denton Hall Stables (M. Dods),Lot Withdrawn,NaN,NaN,2020,Autumn Horses In Training Sale 2020,-,2017LETHAL FORCEJOYEAUX,LETHAL FORCE,JOYEAUX,MARK OF ESTEEM,DENTON HALL STABLES
3,2,764,Summerghand (IRE),G,B,Lope de Vega (IRE),Kate The Great (GB),"2,014.00",6/3/2014,Shamardal (USA),Xaar (GB),NaN,David O'Meara Racing Ltd.,Vendor,90.000,NaN,2020,Autumn Horses In Training Sale 2020,108.675,2014LOPE DE VEGAKATE THE GREAT,LOPE DE VEGA,KATE THE GREAT,XAAR,DAVID O'MEARA RACING LTD.
4,2,691,Regal Reality (GB),G,B,Intello (GER),Regal Realm (GB),"2,015.00",20/2/2015,Galileo (IRE),Medicean (GB),NaN,Freemason Lodge Stables (Sir M. Stoute),Lot Withdrawn,NaN,NaN,2020,Autumn Horses In Training Sale 2020,-,2015INTELLOREGAL REALM,INTELLO,REGAL REALM,MEDICEAN,FREEMASON LODGE STABLES


## 2. Column Standardisation

Three sequential passes:
1. Strip leading/trailing whitespace from column names.
2. Lowercase all names for uniform access.
3. Rename domain-specific labels to unambiguous `snake_case` identifiers
   (e.g. `Año Subasta` → `sale_year`, `Price (gns)` → `price_gns`).

In [4]:
# Normalise column names
print("Before normalising names:",autumn_horses_df.columns)
col_names_to_normalise = [col for col in autumn_horses_df.columns if col.strip() != col]
for col_name in col_names_to_normalise:
    autumn_horses_df[col_name.strip()] = autumn_horses_df[col_name]
    autumn_horses_df = autumn_horses_df.drop(columns=[col_name])
print("After normalising names:",autumn_horses_df.columns)

Before normalising names: Index(['Day', 'Lot', 'Name', 'Sex', 'Colour', 'Sire', 'Dam', 'Year Foaled',
       'Date Foaled', 'Grandsire', 'Damsire', 'Covered by', 'Consignor',
       'Purchaser', 'Price (gns)', 'Stabling', 'Año Subasta', 'Nombre Subasta',
       ' Price (€)', ' ORIG', ' SIRE_N', ' DAM_N', ' SIREDAM_N', ' BREEDER_N'],
      dtype='object')
After normalising names: Index(['Day', 'Lot', 'Name', 'Sex', 'Colour', 'Sire', 'Dam', 'Year Foaled',
       'Date Foaled', 'Grandsire', 'Damsire', 'Covered by', 'Consignor',
       'Purchaser', 'Price (gns)', 'Stabling', 'Año Subasta', 'Nombre Subasta',
       'Price (€)', 'ORIG', 'SIRE_N', 'DAM_N', 'SIREDAM_N', 'BREEDER_N'],
      dtype='object')


## 3. Data Dictionary

The raw file mixes descriptive catalog metadata, commercial outcomes, and already-normalized entity fields. The key point is that some columns are useful for interpretation but dangerous for modeling if they are not framed correctly.
### Core catalog fields
- `Day`: sale day from 1 to 5. It behaves more like catalogue positioning than pure calendar time.
- `Lot`: lot number inside each sale year.
- `Name`: horse name, often with country suffix.
- `Sex`: `C`, `F`, `G`, `H`, `M`, `R`. These labels encode both biological sex and, indirectly, market segment.
- `Colour`: horse colour.
- `Sire`, `Dam`, `Grandsire`, `Damsire`: pedigree fields. They arrive both as raw labels and, for some of them, as normalized canonical versions.
- `Consignor`: seller or consignor label as written in the catalogue.
- `Covered by`: almost empty in this dataset, so analytically weak unless a very specific niche use case is targeted.
- `Stabling`: also extremely sparse, so it should be treated as auxiliary metadata, not as a central predictor.
### Time and price fields
- `Year Foaled`: foaling year.
- `Date Foaled`: full foaling date. It is **not** redundant here; the field is almost complete and contains real month/day variation that can proxy relative maturity.
- `Año Subasta` (original CSV name): sale year. Renamed to `sale_year`.
- `Nombre Subasta` (original CSV name): sale name. Renamed to `sale_name`.
- `Price (gns)`: price in guineas, the most reliable monetary field for analysis.
- `Price (€)`: euro amount embedded in the source file. It should be treated as descriptive only, not as a historically comparable macro-financial series.
### Outcome and normalization fields
- `Purchaser`: buyer label or administrative outcome (`Lot Withdrawn`, `Lot Not Sold`, `Vendor`). These states are economically different and must not be collapsed blindly. 
- `ORIG`: normalized combination of foaling year, sire, and dam. It is useful for auditing but not safe as a unique key.
- `SIRE_N`, `DAM_N`, `SIREDAM_N`, `BREEDER_N`: canonicalized entity labels already supplied by the source. These are preferable to manual row realignment.

In [5]:
col_names_to_lowercase = autumn_horses_df.columns.str.lower()
autumn_horses_df.columns = col_names_to_lowercase
autumn_horses_df.columns

Index(['day', 'lot', 'name', 'sex', 'colour', 'sire', 'dam', 'year foaled',
       'date foaled', 'grandsire', 'damsire', 'covered by', 'consignor',
       'purchaser', 'price (gns)', 'stabling', 'año subasta', 'nombre subasta',
       'price (€)', 'orig', 'sire_n', 'dam_n', 'siredam_n', 'breeder_n'],
      dtype='object')

In [6]:
columns_to_rename = {
    'name': 'horse_name',
    'year foaled': 'birth_year',
    'date foaled': 'date_foaled',
    'price (gns)': 'price_gns',
    'price (€)': 'price_euros_raw',
    'año subasta': 'sale_year',
    'nombre subasta': 'sale_name',
    'covered by': 'stallion',
    'breeder_n': 'consignor_n',
    'siredam_n': 'damsire_n',
}
autumn_horses_df = autumn_horses_df.rename(columns=columns_to_rename)
# Convert price_gns from European string format to numeric (90.000 → 90000.0)
autumn_horses_df['price_gns'] = parse_numeric_series(autumn_horses_df['price_gns'])

## 4. Entity Normalisation & Cleaning

Pedigree and commercial entity fields (sire, dam, grandsire, damsire, consignor) arrive with
country suffixes `(GB)`, typographic noise, and abbreviation variants.
This section:
- Extracts and stores country codes separately (e.g. `sire_country`).
- Applies conservative root-entity normalisation for consignors
  (removes legal suffixes, resolves known aliases such as `JUDDMONTE FARMS → JUDDMONTE`).
- Normalises the catalogue day number to a 0–1 ratio to handle 4- vs 5-day sale editions.

In [7]:
# clean entities without overwriting the original text columns.
# Clean entities without overwriting original columns.
string_columns = [
    'horse_name', 'sex', 'colour', 'sire', 'dam', 'grandsire', 'damsire',
    'stallion', 'consignor', 'purchaser', 'sale_name', 'orig',
    'sire_n', 'dam_n', 'damsire_n', 'consignor_n', 'price_euros_raw'
]
for col in string_columns:
    if col in autumn_horses_df.columns:
        autumn_horses_df[col] = autumn_horses_df[col].astype('string').str.strip()
entity_columns = ['horse_name', 'sire', 'dam', 'grandsire', 'damsire', 'stallion']
for col in entity_columns:
    autumn_horses_df[f'{col}_country'] = extract_country_suffix(autumn_horses_df[col])
    autumn_horses_df[f'{col}_clean'] = strip_country_suffix(autumn_horses_df[col])
autumn_horses_df['horse_name_clean'] = autumn_horses_df['horse_name_clean'].fillna(autumn_horses_df['horse_name'])
autumn_horses_df['consignor_display'] = autumn_horses_df['consignor'].str.replace(r'\s+', ' ', regex=True).str.strip()
autumn_horses_df['sire_entity'] = autumn_horses_df['sire_n']
autumn_horses_df['dam_entity'] = autumn_horses_df['dam_n']
autumn_horses_df['damsire_entity'] = autumn_horses_df['damsire_n']
autumn_horses_df['consignor_entity_exact'] = autumn_horses_df['consignor_n'].fillna(autumn_horses_df['consignor'].str.upper())
autumn_horses_df['consignor_entity'] = autumn_horses_df['consignor_entity_exact']
consignor_stopwords = {'LTD', 'LIMITED', 'LLP', 'INC', 'COMPANY', 'CO', 'AGENT', 'IRELAND', 'STABLES', 'STABLE', 'RACING'}
consignor_aliases = {'JUDDMONTE FARMS': 'JUDDMONTE'}
autumn_horses_df['sire_clean'] = autumn_horses_df['sire_clean'].fillna(title_from_canonical(autumn_horses_df['sire_entity']))
autumn_horses_df['dam_clean'] = autumn_horses_df['dam_clean'].fillna(title_from_canonical(autumn_horses_df['dam_entity']))
autumn_horses_df['damsire_clean'] = autumn_horses_df['damsire_clean'].fillna(title_from_canonical(autumn_horses_df['damsire_entity']))
autumn_horses_df['consignor_label'] = title_from_canonical(autumn_horses_df['consignor_entity_exact']).fillna(autumn_horses_df['consignor_display'])
# Day Normalization (Equivalence 4 vs 5 days)
# Logic: Map days to a 0.0 - 1.0 scale within each year to ensure comparability.
def get_day_norm_ratio(day_series):
    max_day = day_series.max()
    if max_day <= 1: 
        return 0.0
    return (day_series - 1) / (max_day - 1)
autumn_horses_df['day_norm_ratio'] = autumn_horses_df.groupby('sale_year')['day'].transform(get_day_norm_ratio)
# To keep it categorical and readable: "Day 1", "Day 2", "Day 3", "Day 4"
# We'll use a 4-bin approach or just handle the 5th day as "Late Sale"
autumn_horses_df['day_normalized'] = np.round(autumn_horses_df['day_norm_ratio'] * 3 + 1).astype(int)
autumn_horses_df['consignor_family'] = autumn_horses_df['consignor_entity_exact'].map(
    lambda x: normalize_root_entity(x, stopwords=consignor_stopwords, aliases=consignor_aliases)
).fillna(autumn_horses_df['consignor_label'])
autumn_horses_df['consignor_clean'] = autumn_horses_df['consignor_label']
autumn_horses_df['consignor_model'] = autumn_horses_df['consignor_entity_exact']
autumn_horses_df['birth_year'] = autumn_horses_df['birth_year'].astype('Int64')
autumn_horses_df['sale_year'] = autumn_horses_df['sale_year'].astype('Int64')
autumn_horses_df['date_foaled'] = pd.to_datetime(autumn_horses_df['date_foaled'], dayfirst=True, errors='coerce')
autumn_horses_df[[
    'horse_name', 'horse_name_clean', 'horse_name_country',
    'grandsire', 'grandsire_clean', 'grandsire_country',
    'consignor', 'consignor_label', 'consignor_family'
]].head()

,horse_name,horse_name_clean,horse_name_country,grandsire,grandsire_clean,grandsire_country,consignor,consignor_label,consignor_family
0,Qirat (GB),Qirat,GB,Oasis Dream (GB),Oasis Dream,GB,Juddmonte,Juddmonte,JUDDMONTE
1,Gifted Master (IRE),Gifted Master,IRE,Danehill (USA),Danehill,USA,The Castlebridge Consignment,The Castlebridge Consignment,THE CASTLEBRIDGE CONSIGNMENT
2,Commanche Falls (GB),Commanche Falls,GB,Dark Angel (IRE),Dark Angel,IRE,Denton Hall Stables (M. Dods),Denton Hall Stables,DENTON HALL
3,Summerghand (IRE),Summerghand,IRE,Shamardal (USA),Shamardal,USA,David O'Meara Racing Ltd.,David O'Meara Racing Ltd.,DAVID O MEARA
4,Regal Reality (GB),Regal Reality,GB,Galileo (IRE),Galileo,IRE,Freemason Lodge Stables (Sir M. Stoute),Freemason Lodge Stables,FREEMASON LODGE


## 5. Derived Features

All derived columns are additive — no source column is deleted. Key derivations:
- `age_at_sale`: sale year minus birth year.
- `foaled_month` / `foaled_quarter`: seasonality proxy for relative maturity within a cohort.
- `is_late_catalogue_day`: boolean flag for catalogue positions on days 4–5.
- `sire_dam_combo`: cross-reference key for pedigree novelty scoring in Feature Engineering.

In [8]:
# derive features that are analytically useful and reproducible.
autumn_horses_df['age_at_sale'] = autumn_horses_df['sale_year'] - autumn_horses_df['birth_year']
autumn_horses_df['foaled_month'] = autumn_horses_df['date_foaled'].dt.month.astype('Int64')
autumn_horses_df['foaled_quarter'] = autumn_horses_df['date_foaled'].dt.quarter.astype('Int64')
autumn_horses_df['is_late_catalogue_day'] = autumn_horses_df['day'].isin([4, 5])
autumn_horses_df['sire_dam_combo'] = autumn_horses_df['sire_entity'].fillna('UNKNOWN') + '_x_' + autumn_horses_df['dam_entity'].fillna('UNKNOWN')
missing_summary = (
    autumn_horses_df.isna()
    .mean()
    .sort_values(ascending=False)
    .rename('missing_share')
    .to_frame()
)
missing_summary.head(12)

,missing_share
stallion_clean,1.00
stallion,1.00
stallion_country,1.00
stabling,0.98
price_gns,0.28
damsire_country,0.08
grandsire_country,0.05
sire_country,0.00
dam_country,0.00
damsire,0.00


## 6. Outcome Engineering

The `sale_outcome` variable encodes **five mutually exclusive states**:

| Outcome | Description |
|---------|-------------|
| `sold_to_third_party` | Genuine market transaction — the target for price modelling |
| `vendor_buyback` | Reserve met by the vendor; price recorded but no real transfer |
| `not_sold` | Failed to meet any bid at or above reserve |
| `withdrawn` | Never entered the ring; excluded from sell-through denominators |
| `other_or_inconsistent` | Data anomaly flag (should be 0 after cleaning) |

The distinction between `vendor_buyback` and `not_sold` is critical: collapsing them
would inflate apparent clearance rates and introduce upward price bias.

In [9]:
# define outcomes explicitly instead of collapsing all non-sales together.
# Define outcomes explicitly instead of collapsing all non-sales.
withdrawn_mask = autumn_horses_df['purchaser'].eq('Lot Withdrawn')
not_sold_mask = autumn_horses_df['purchaser'].eq('Lot Not Sold')
vendor_mask = autumn_horses_df['purchaser'].eq('Vendor')
sold_to_third_party_mask = (~withdrawn_mask & ~not_sold_mask & ~vendor_mask) & autumn_horses_df['price_gns'].notna()
autumn_horses_df['sale_outcome'] = np.select(
    [withdrawn_mask, not_sold_mask, vendor_mask, sold_to_third_party_mask],
    ['withdrawn', 'not_sold', 'vendor_buyback', 'sold_to_third_party'],
    default='other_or_inconsistent'
)
autumn_horses_df['sale_outcome'] = pd.Categorical(
    autumn_horses_df['sale_outcome'],
    categories=['sold_to_third_party', 'vendor_buyback', 'not_sold', 'withdrawn', 'other_or_inconsistent'],
    ordered=False,
)
autumn_horses_df['has_price_quote'] = autumn_horses_df['price_gns'].notna()
autumn_horses_df['is_offered_for_sale'] = ~withdrawn_mask
autumn_horses_df['sold_to_third_party'] = sold_to_third_party_mask
autumn_horses_df['vendor_buyback'] = vendor_mask
autumn_horses_df['lot_not_sold'] = not_sold_mask
autumn_horses_df['lot_withdrawn'] = withdrawn_mask
autumn_horses_df['sold'] = autumn_horses_df['sold_to_third_party']
autumn_horses_df['log_price_gns'] = np.nan
positive_log_mask = autumn_horses_df['sold_to_third_party'] & autumn_horses_df['price_gns'].gt(0)
autumn_horses_df.loc[positive_log_mask, 'log_price_gns'] = np.log(
    autumn_horses_df.loc[positive_log_mask, 'price_gns']
)
autumn_horses_df['sale_outcome'].value_counts(dropna=False)

sale_outcome
sold_to_third_party      16531
withdrawn                 7081
vendor_buyback            1383
not_sold                  1081
other_or_inconsistent        0
Name: count, dtype: int64

## 7. Temporal Sanity Check

Annual aggregate statistics serve two purposes:
1. A final consistency check — confirming that derived flags sum correctly across years.
2. A summary table that feeds directly into Section 2 of the EDA notebook.

`sale_rate_on_offered` (sold to third party ÷ lots offered) is the primary clearance KPI
used throughout the TFM. `withdrawal_rate` tracks the pre-ring withdrawal trend.

In [10]:
# 2. Temporal structure
by_year = (
    autumn_horses_df.groupby('sale_year')
    .agg(
        total_catalogued=('lot', 'size'),
        offered=('is_offered_for_sale', 'sum'),
        sold_to_third_party=('sold_to_third_party', 'sum'),
        vendor_buyback=('vendor_buyback', 'sum'),
        lot_not_sold=('lot_not_sold', 'sum'),
        withdrawn=('lot_withdrawn', 'sum'),
        median_price_sold=('price_gns', lambda s: s[autumn_horses_df.loc[s.index, 'sold_to_third_party']].median())
    )
)
by_year['sale_rate_on_catalogue'] = 100 * by_year['sold_to_third_party'] / by_year['total_catalogued']
by_year['sale_rate_on_offered'] = 100 * by_year['sold_to_third_party'] / by_year['offered']
by_year['withdrawal_rate'] = 100 * by_year['withdrawn'] / by_year['total_catalogued']
# Merge median_price_real if price_real_gns already exists (defined in cell 26 onward)
if 'price_real_gns' in autumn_horses_df.columns:
    _real_median = (
        autumn_horses_df[autumn_horses_df['sold_to_third_party']]
        .groupby('sale_year')['price_real_gns']
        .median()
        .rename('median_price_real')
    )
    by_year = by_year.join(_real_median)
by_year.round(2)

,total_catalogued,offered,sold_to_third_party,vendor_buyback,lot_not_sold,withdrawn,median_price_sold,sale_rate_on_catalogue,sale_rate_on_offered,withdrawal_rate
sale_year,,,,,,,,,,
2009,1533,1063,903,69,91,470,9000,58.90,84.95,30.66
2010,1583,1087,865,88,134,496,9000,54.64,79.58,31.33
2011,1478,1020,848,80,92,458,9000,57.37,83.14,30.99
2012,1429,1015,909,69,37,414,11000,63.61,89.56,28.97
2013,1518,1088,894,119,75,430,10000,58.89,82.17,28.33
2014,1539,1047,922,78,47,492,13000,59.91,88.06,31.97
2015,1679,1220,1038,92,90,459,10000,61.82,85.08,27.34
2016,1516,1032,949,52,31,484,13500,62.60,91.96,31.93
2017,1650,1255,1065,111,79,395,11000,64.55,84.86,23.94


In [11]:
df_sold = autumn_horses_df.loc[autumn_horses_df['sold_to_third_party']].copy()


## 8. Export

A single Parquet file — `data/processed/clean_data.parquet` — is the source of truth
for all downstream notebooks. Parquet preserves dtypes (including `Int64` nullable integers
and `Categorical`) without round-trip loss.

In [12]:
os.makedirs('../data/processed', exist_ok=True)
autumn_horses_df.to_parquet('../data/processed/clean_data.parquet', index=False)
print('Data saved to data/processed/clean_data.parquet')

Data saved to data/processed/clean_data.parquet
